## Creating a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits

In [2]:
import os
import json
from dotenv import load_dotenv


In [3]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

In [4]:
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

API key looks good


In [5]:
from openai import OpenAI

openai = OpenAI()

In [6]:
#fetch links from the website - https://huggingface.co"
from scraper import fetch_website_links

link = "https://www.anthropic.com/"

links = fetch_website_links(link)

links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/economic-futures',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'http://trust.anthropic.com/',
 'https://www.anthropic.com/learn',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://www.anthropic.com/engineering',
 'https://platform.claude.com/docs',
 'https://www.anthropic.com/company',
 'https://www.anthropic.com/careers',
 'https://www.anthropic.com/events',
 'https://www.anthropic.com/news',
 'https://claude.ai',
 'https://claude.com/product/overview',
 'https://claude.com/product/claude-code',
 'https://claude.com/product/cowork',
 'https://claude.com/product/claude-security',
 'https://claude.com/platform/api',
 'https://claude.com/pricing',
 'https://claude.com/contact-sales',
 'https://www.anthropic.com/claude/opus',
 

### We get all the links from the website, these links includes business links and some links which are not relevent to the website 


### We need to Make sure, we get those links that are relevent to create a brochure using AI

## Let's do it

## First Step - Have GPT-5-nano figure out Which Links are relevant

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    """
    links = fetch_website_links(link)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt("https://www.anthropic.com/"))


    Here is the list of links on the website https://www.anthropic.com/ -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    #main
#footer
https://www.anthropic.com/
https://www.anthropic.com/research
https://www.anthropic.com/economic-futures
https://www.anthropic.com/constitution
https://www.anthropic.com/transparency
https://www.anthropic.com/responsible-scaling-policy
http://trust.anthropic.com/
https://www.anthropic.com/learn
https://claude.com/resources/tutorials
https://claude.com/resources/use-cases
https://www.anthropic.com/engineering
https://platform.claude.com/docs
https://www.anthropic.com/company
https://www.anthropic.com/careers
https://www.anthropic.com/events
https://www.anthropic.com/news
https://claude.ai
https://claude.com/product/overview
https://claude.com/product/cla

In [10]:
MODEL = "gpt-5-nano"

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create( model=MODEL, messages=[
        { "role":"system", "content": link_system_prompt},
        {"role":"user", "content": get_links_user_prompt(url)
        }],
        response_format={"type" : "json_object"}
        )
    results = response.choices[0].message.content
    links = json.loads(results)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [23]:
select_relevant_links("https://www.anthropic.com/")

Selecting relevant links for https://www.anthropic.com/ by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'about page', 'url': 'https://www.anthropic.com/company'},
  {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'research page', 'url': 'https://www.anthropic.com/research'},
  {'type': 'news page', 'url': 'https://www.anthropic.com/news'},
  {'type': 'events page', 'url': 'https://www.anthropic.com/events'},
  {'type': 'engineering page', 'url': 'https://www.anthropic.com/engineering'},
  {'type': 'learn resources page', 'url': 'https://www.anthropic.com/learn'},
  {'type': 'transparency page',
   'url': 'https://www.anthropic.com/transparency'},
  {'type': 'constitution / governance page',
   'url': 'https://www.anthropic.com/constitution'},
  {'type': 'economic futures page',
   'url': 'https://www.anthropic.com/economic-futures'},
  {'type': ' Claude product overview (external)',
   'url': 'https://claude.com/product/overview'}]}

## Second Step - Let's Make the Brochure

In [12]:
from scraper import fetch_website_contents


def make_brochure(url):
    contents = fetch_website_contents(url)
    links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [28]:
print(make_brochure("https://www.anthropic.com/"))

Selecting relevant links for https://www.anthropic.com/ by calling gpt-5-nano
Found 16 relevant links
## Landing Page:

Home \ Anthropic

Skip to main content
Skip to footer
Research
Economic Futures
Commitments
Initiatives
Claude's Constitution
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
Products
Claude
Claude Code
Claude Cowork
Claude Security
Claude Platform
Pricing
Contact sales
Models
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
Research
Economic Futures
Commitments
Initiatives
Claude's Constitution
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineerin

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += make_brochure(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
from IPython.display import Markdown, display

def create_brochure(company_name, url):
    response = openai.chat.completions.create(model=Model, messages=[
        {"role" : "system", "content":brochure_system_prompt },
        {"role":"user", "content" : get_brochure_user_prompt(company_name, url)},
        ])
    result = response.choices[0].message.content
    display(Markdown(result))

In [32]:
create_brochure("Antropic", "https://www.anthropic.com/")

Selecting relevant links for https://www.anthropic.com/ by calling gpt-5-nano
Found 12 relevant links


Anthropic: AI safety and research at the frontier

Overview
- Anthropic is an AI safety and research company structured as a public benefit corporation. Our mission is to build AI systems that people can rely on, by prioritizing safety, reliability, interpretability, and steerability.
- We believe AI will have a vast impact on the world, and we focus on creating frontier AI that delivers benefits while actively mitigating risks.

What we do
- Build safer AI systems: frontier research, safety techniques, and practical deployments of reliable, interpretable, and controllable AI.
- Develop a family of Claude-powered products and platforms designed for developers, enterprises, and researchers:
  - Claude: core AI assistant with a focus on helpful, thoughtful conversations
  - Claude Code, Claude Cowork, Claude Security, Claude Platform
  - Claude.ai and Claude Console for interaction and management
- Push the boundaries of safety, governance, and transparency in AI, with a strong emphasis on responsible scaling and risk mitigation.

Technology highlights
- Latest releases and capabilities: Claude Opus 4.8, an upgrade across coding, agentic tasks, and professional work, designed to handle long-running workflows with consistency.
- Claude is described as “a space to think”—no ads, no sponsored content, just genuinely helpful conversations.
- Ongoing initiatives include Project Glasswing, aimed at securing critical software for the AI era.

Models and platforms
- Models: Opus, Sonnet, Haiku
- Products and services: Claude, Claude Code, Claude Cowork, Claude Security, Claude Platform
- Access and management: Claude Console, Claude.ai; pricing and trials available for customers and developers

Governance, safety, and trust
- Claude’s Constitution: a framework for aligning AI behavior with safety and ethical considerations
- Transparency: commitments and information to help users understand how models work
- Responsible Scaling Policy: guidelines for safely expanding model capabilities
- Trust Center and Security & Compliance: emphasis on securing data, privacy, and compliance
- A culture and infrastructure built around safe deployment and reliability

Learn and grow
- Education and resources to help teams design with safety in mind:
  - Anthropic Academy, Tutorials, Use cases
  - Engineering at Anthropic and Developer docs
- A commitment to research-driven improvement and practical guidance for building with Claude

Careers and culture
- A mission-driven, safety-focused culture motivated by building AI you can rely on
- The site highlights opportunities to join, with a dedicated Careers page
- Roles span research, engineering, policy, security, and product areas, all contributing to safer AI systems
- Community and events: opportunities to engage with the broader Anthropic ecosystem through news, events, and ongoing learning

Customers, partners, and engagement
- Claude and the Claude ecosystem are positioned for developers, organizations, and researchers seeking reliable, steerable AI solutions
- Engagement channels include Try Claude, product demos, and contact for sales
- The emphasis is on practical deployment of safe AI that people can rely on in real-world settings

Why partner with Anthropic
- A public benefit corporation focused on long-term societal benefit, balancing innovation with safety and risk mitigation
- A proven commitment to building safer, more interpretable, and steerable AI systems
- Ongoing research, transparent governance, and robust security practices designed to foster trust and reliability

How to get started
- Try Claude to experience the capabilities firsthand
- Explore the Claude suite (Core Claude, Code, Cowork, Security, Platform) and related models (Opus, Sonnet, Haiku)
- Learn about safety, governance, and transparency resources through Claude’s Constitution, Transparency materials, and the Trust Center
- Reach out via sales for pricing, trials, and enterprise engagements

Key quotes and spirit
- “AI will have a vast impact on the world. Anthropic is a public benefit corporation dedicated to securing its benefits and mitigating its risks.”
- “Claude is a space to think. No ads. No sponsored content. Just genuinely helpful conversations.”
- “Making AI systems you can rely on” — Anthropic’s core purpose and approach to safety, reliability, and steerability

This brochure highlights the core elements visible on Anthropic’s landing and about pages: mission, safety-forward AI development, the Claude product ecosystem, governance and transparency practices, learning resources, and career opportunities. If you’d like, I can tailor this into a one-page PDF brochure or a slide deck outline for investors, customers, or recruits.

## Lets improvise it with Streaming - instead of waiting for the full response, you display it word by word like ChatGPT does

In [ ]:

from IPython.display import update_display

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(model = MODEL, messages=[
        {"role" : "system", "content":brochure_system_prompt },
        {"role":"user", "content" : get_brochure_user_prompt(company_name, url)},],
        stream=True
    )
    response = ""

    # Creates an empty placeholder in the notebook output. display_id=True gives it a unique ID so you can update it later.
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:

        # Each chunk contains a small piece of text (could be one word or few characters). .delta means "the new piece
        response += chunk.choices[0].delta.content or ''

        # Updates that same placeholder with the growing response on every chunk
        update_display(Markdown(response), display_id=display_handle.display_id)



In [37]:
stream_brochure("Antropic", "https://www.anthropic.com/")

Selecting relevant links for https://www.anthropic.com/ by calling gpt-5-nano
Found 11 relevant links


Anthropic: AI safety and research for a responsible future

Overview
- Anthropic is an AI safety and research company dedicated to building reliable, interpretable, and steerable AI systems.
- We are a public benefit corporation committed to securing AI benefits and mitigating risks.
- Our work spans frontier research, practical safety techniques, and products that help people work with powerful AI responsibly.

What we do
- Build and deploy safer AI systems that people can rely on.
- Conduct frontier AI research across modalities and safety areas, from interpretability to learning from human feedback, policy, and societal impact analysis.
- Create practical products and platforms that emphasize safety, reliability, and accessibility.

Key products and platforms
- Claude family: Claude, Claude Code, Claude Cowork, Claude Security, Claude Platform, Claude Console, Claude.ai
- Models: Opus, Sonnet, Haiku
- Tools and resources: Claude Academy, Tutorials, Use cases, Developer docs
- Claude experience: No ads, no sponsored content—designed to be a space to think and have genuinely helpful conversations
- Recent updates: Claude Opus 4.8 (enhanced coding, agentic tasks, and long-running workflows)

How we design and govern AI
- Claude’s Constitution and other governance work to align AI behavior with human values
- Transparency: sharing insights and methodologies to advance responsible AI
- Responsible Scaling Policy and Trust Center: governance and safety practices for scaling AI
- Security and compliance as foundational elements
- Safety as a science: systematic research, application to products, and sharing learnings with the world

Culture and approach
- Interdisciplinary teamwork: researchers, engineers, policy experts, and operators from diverse disciplines
- We strive to build frontier AI systems that are reliable, interpretable, and steerable
- Collaboration with a broad ecosystem: civil society, government, academia, nonprofits, and industry to promote safety industry-wide
- Our people are motivated by hard problems with real stakes and a mission to shape how AI meets the world

Customers, partners, and impact
- We collaborate with a wide range of stakeholders to ensure AI benefits are realized safely and responsibly
- Project Glasswing highlights our commitment to securing critical software for the AI era
- We view AI as part of a larger puzzle and work with partners across sectors to advance safety and responsible deployment

Careers and joining Anthropic
- Shape how AI meets the world by building Claude—an AI designed to be helpful, honest, and harmless
- We recruit researchers, engineers, policy experts, and operational leaders from diverse backgrounds
- If you’re drawn to hard problems with real stakes, we’d like to meet you
- Explore open roles and learn more about building Anthropic

How to engage and learn more
- Try Claude and explore Claude-related products (Claude, Claude Code, Claude Platform, Claude Security)
- Learn through Anthropic Academy, Tutorials, Use cases, and Developer docs
- Stay connected with our research, news, and events pages
- Contact sales for product-specific inquiries or partnerships

Summary
Anthropic is an AI safety and research company focused on creating reliable, understandable, and steerable AI systems. Through rigorous safety science, interdisciplinary collaboration, and a broad ecosystem of partners, we aim to maximize AI benefits while responsibly mitigating risks. We seek curious, mission-driven individuals to join us in shaping how AI serves the global good.

## Gradio for Brochure

In [1]:
import gradio as gr

/Users/kavyahj/AI-Projects/brochure-generator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
def display_brochure(company_name, url):
    messages = [
        {"role":"system", "content" : brochure_system_prompt},
        {"role":"user", "content" : get_brochure_user_prompt(company_name, url)}
    ]

    response = openai.chat.completions.create(model = "gpt-5-nano", messages=messages, stream = True)
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [20]:
message_Input = gr.Textbox(label = "Enter the Company Name")
message_Input2 = gr.Textbox(label = "Enter the Company Website URL")
message_output = gr.Markdown(label = "Response")

gr.Interface(
    fn = display_brochure,
    title = "Brochure Generate using GPT",
    inputs = [message_Input, message_Input2],
    outputs = [message_output],
    flagging_mode = "never"
).launch()


* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://huggingface.co/ by calling gpt-5-nano
Found 12 relevant links
